# Brief A — Test Volume Growth Investigation
## 01 | CONFIG — Environment & Analysis Periods

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve().parent
RAW = ROOT / "data" / "raw"

print("Data folder exists:", RAW.exists())
print("Available folders:")
print([p.name for p in RAW.iterdir() if p.is_dir()])

Data folder exists: True
Available folders:
['patients', 'payments', 'bookings', 'invoices', 'plan', 'results', 'campaigns', 'collections', 'reference']


## 02 | DATA INVENTORY — Identify Source Files

**Objective:** Identify the raw datasets relevant to the test-volume investigation before loading or transforming them.

In [2]:
brief_a_folders = [
    "patients",
    "bookings",
    "collections",
    "results"
]

for folder in brief_a_folders:
    path = RAW / folder

    print(f"\n{'=' * 45}")
    print(f"DATASET: {folder.upper()}")
    print(f"{'=' * 45}")

    if not path.exists():
        print("Folder missing")
        continue

    files = sorted(
        p for p in path.rglob("*")
        if p.is_file()
    )

    print("Number of files:", len(files))

    for file in files:
        print(" -", file.relative_to(RAW))


DATASET: PATIENTS
Number of files: 1
 - patients/patients.csv.gz

DATASET: BOOKINGS
Number of files: 108
 - bookings/legacy/ahmedabad/bookings_2025-01.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-02.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-03.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-04.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-05.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-06.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-07.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-08.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-09.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-10.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-11.csv.gz
 - bookings/legacy/ahmedabad/bookings_2025-12.csv.gz
 - bookings/legacy/ahmedabad/bookings_2026-01.csv.gz
 - bookings/legacy/ahmedabad/bookings_2026-02.csv.gz
 - bookings/legacy/ahmedabad/bookings_2026-03.csv.gz
 - bookings/legacy/ahmedabad/bookings_2026-04.csv.gz
 - bookings/legacy/ahmedabad/bookings_2026-05.

## 03 | PROFILING — Data Structure & Quality

**Objective:** Inspect row counts, column types, missing values, duplicates and date coverage before calculating business metrics.

In [3]:
# Inspect one sample file from each source folder
sample_files = {}

for folder in brief_a_folders:
    files = sorted((RAW / folder).rglob("*.csv.gz"))

    if not files:
        print(f"{folder}: No CSV files found")
        continue

    sample_files[folder] = files[0]

    print(f"\n{'=' * 55}")
    print(f"DATASET: {folder.upper()}")
    print("File:", files[0].relative_to(RAW))

    try:
        sample = pd.read_csv(files[0], nrows=5)
    except UnicodeDecodeError:
        sample = pd.read_csv(
            files[0], nrows=5, encoding="latin-1"
        )
        print("Encoding fallback: latin-1")

    print("Columns:", sample.columns.tolist())
    print("Shape:", sample.shape)
    print("\nSample records:")
    display(sample.head(3))


DATASET: PATIENTS
File: patients/patients.csv.gz
Columns: ['patient_id', 'patient_name', 'phone', 'dob', 'city', 'signup_date']
Shape: (5, 6)

Sample records:


,patient_id,patient_name,phone,dob,city,signup_date
0,PAHM0030088,Rahul Fernandes,+91 85778 98441,1988-02-02,Ahmedabad,2023-01-14
1,PAHM0043743,Ananya D'Souza,+91 84824 41827,1992-11-13,Ahmedabad,2025-10-02
2,PAHM0011366,Rohan Mehta,+91 77327 81379,1944-07-05,Ahmedabad,2023-01-18



DATASET: BOOKINGS
File: bookings/legacy/ahmedabad/bookings_2025-01.csv.gz
Columns: ['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel', 'source', 'patient_id', 'patient_name', 'test_code', 'test_price', 'home_fee']
Shape: (5, 12)

Sample records:


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee
0,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0
1,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0
2,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0



DATASET: COLLECTIONS
File: collections/collections_AHM_2025-01.csv.gz
Columns: ['booking_id', 'clinic_id', 'channel', 'status', 'collected_at', 'phlebotomist_id']
Shape: (5, 6)

Sample records:


,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id
0,BK-AHM-202501-000001,AHM-03,clinic,collected,2025-01-09 09:09:00,NaN
1,BK-AHM-202501-000002,AHM-04,clinic,collected,2025-01-06 13:44:00,NaN
2,BK-AHM-202501-000003,AHM-01,home,collected,2025-01-28 13:37:00,PHAHM047



DATASET: RESULTS
File: results/ahmedabad/results_2025-01.csv.gz
Columns: ['booking_id', 'test_code', 'result_ts', 'result_value', 'result_flag']
Shape: (5, 5)

Sample records:


,booking_id,test_code,result_ts,result_value,result_flag
0,BK-AHM-202501-000001,T009,2025-01-09 20:09:00,46.8,normal
1,BK-AHM-202501-000001,T004,2025-01-10 00:09:00,49.0,normal
2,BK-AHM-202501-000001,T007,2025-01-09 18:09:00,74.8,normal


In [4]:
# Inspect available schemas without loading all files

for folder in brief_a_folders:
    files = sorted((RAW / folder).rglob("*.csv.gz"))

    print(f"\n{'=' * 55}")
    print(f"DATASET: {folder.upper()}")
    print("Total files:", len(files))

    if not files:
        continue

    for file in files[:2]:
        try:
            df_sample = pd.read_csv(file, nrows=100)
        except UnicodeDecodeError:
            df_sample = pd.read_csv(
                file, nrows=100, encoding="latin-1"
            )

        print("\nFile:", file.relative_to(RAW))
        print("Sample shape:", df_sample.shape)
        print("Columns:", df_sample.columns.tolist())
        print("Missing values:")
        print(df_sample.isna().sum().to_dict())
        print("Duplicate sample rows:", df_sample.duplicated().sum())


DATASET: PATIENTS
Total files: 1

File: patients/patients.csv.gz
Sample shape: (100, 6)
Columns: ['patient_id', 'patient_name', 'phone', 'dob', 'city', 'signup_date']
Missing values:
{'patient_id': 0, 'patient_name': 0, 'phone': 0, 'dob': 0, 'city': 0, 'signup_date': 0}
Duplicate sample rows: 0

DATASET: BOOKINGS
Total files: 102

File: bookings/legacy/ahmedabad/bookings_2025-01.csv.gz
Sample shape: (100, 12)
Columns: ['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel', 'source', 'patient_id', 'patient_name', 'test_code', 'test_price', 'home_fee']
Missing values:
{'booking_id': 0, 'created_at': 0, 'slot_at': 0, 'city': 0, 'clinic_id': 0, 'channel': 0, 'source': 0, 'patient_id': 0, 'patient_name': 0, 'test_code': 0, 'test_price': 0, 'home_fee': 0}
Duplicate sample rows: 0

File: bookings/legacy/ahmedabad/bookings_2025-02.csv.gz
Sample shape: (100, 12)
Columns: ['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel', 'source', 'patient_id', 'patient_nam

In [5]:
for folder in brief_a_folders:
    files = sorted((RAW / folder).rglob("*.csv.gz"))

    if not files:
        continue

    try:
        df_sample = pd.read_csv(files[0], nrows=100)
    except UnicodeDecodeError:
        df_sample = pd.read_csv(
            files[0], nrows=100, encoding="latin-1"
        )

    print(f"\n{folder.upper()}")

    for column in df_sample.columns:
        if any(
            word in column.lower()
            for word in ["date", "time", "status", "id", "test"]
        ):
            print(
                column,
                "| dtype:", df_sample[column].dtype,
                "| examples:",
                df_sample[column].dropna().unique()[:5]
            )


PATIENTS
patient_id | dtype: str | examples: <ArrowStringArray>
['PAHM0030088', 'PAHM0043743', 'PAHM0011366', 'PAHM0009021', 'PAHM0052680']
Length: 5, dtype: str
signup_date | dtype: str | examples: <ArrowStringArray>
['2023-01-14', '2025-10-02', '2023-01-18', '2022-08-06', '2026-05-19']
Length: 5, dtype: str

BOOKINGS
booking_id | dtype: str | examples: <ArrowStringArray>
['BK-AHM-202501-000001', 'BK-AHM-202501-000002', 'BK-AHM-202501-000003',
 'BK-AHM-202501-000004', 'BK-AHM-202501-000005']
Length: 5, dtype: str
clinic_id | dtype: str | examples: <ArrowStringArray>
['AHM-03', 'AHM-04', 'AHM-01', 'AHM-05', 'AHM-02']
Length: 5, dtype: str
patient_id | dtype: str | examples: <ArrowStringArray>
['PAHM0028525', 'PAHM0031384', 'PAHM0027001', 'PAHM0007069', 'PAHM0002520']
Length: 5, dtype: str
test_code | dtype: str | examples: <ArrowStringArray>
['T009', 'T004', 'T007', 'T003', 'T017']
Length: 5, dtype: str
test_price | dtype: int64 | examples: [800 650 550 450 120]

COLLECTIONS
booking_i

### 03.1 | Schema Consistency — Legacy vs Current Files

In [6]:
for folder in ["bookings", "collections", "results"]:
    files = sorted((RAW / folder).rglob("*.csv.gz"))
    schemas = {}

    for file in files:
        try:
            columns = tuple(pd.read_csv(file, nrows=0).columns)
        except UnicodeDecodeError:
            columns = tuple(
                pd.read_csv(file, nrows=0, encoding="latin-1").columns
            )

        schemas.setdefault(columns, []).append(file)

    print(f"\n{folder.upper()}: {len(files)} files")
    print("Different schemas:", len(schemas))

    for i, (columns, matching_files) in enumerate(
        schemas.items(), 1
    ):
        print(f"Schema {i}: {len(matching_files)} files")
        print("Example:", matching_files[0].relative_to(RAW))
        print("Columns:", list(columns))


BOOKINGS: 102 files
Different schemas: 1
Schema 1: 102 files
Example: bookings/legacy/ahmedabad/bookings_2025-01.csv.gz
Columns: ['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel', 'source', 'patient_id', 'patient_name', 'test_code', 'test_price', 'home_fee']

COLLECTIONS: 108 files
Different schemas: 1
Schema 1: 108 files
Example: collections/collections_AHM_2025-01.csv.gz
Columns: ['booking_id', 'clinic_id', 'channel', 'status', 'collected_at', 'phlebotomist_id']

RESULTS: 108 files
Different schemas: 1
Schema 1: 108 files
Example: results/ahmedabad/results_2025-01.csv.gz
Columns: ['booking_id', 'test_code', 'result_ts', 'result_value', 'result_flag']


### 03.2 | Data Quality Checks — Missing Values, Duplicates & Identifiers

**Objective:** Check source records for missing identifiers, duplicate rows and data-quality issues before calculating test volume.

In [7]:
# Read source files and check their quality without changing them

datasets = {}

for folder in ["patients", "bookings", "collections", "results"]:
    files = sorted((RAW / folder).rglob("*.csv.gz"))

    frames = []

    for file in files:
        try:
            df_part = pd.read_csv(file)
        except UnicodeDecodeError:
            df_part = pd.read_csv(file, encoding="latin-1")

        frames.append(df_part)

    if not frames:
        print(f"{folder}: No files found")
        continue

    df = pd.concat(frames, ignore_index=True)
    datasets[folder] = df

    print(f"\n{'=' * 45}")
    print(folder.upper())
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Missing booking_id:",
          df["booking_id"].isna().sum()
          if "booking_id" in df.columns else "Not applicable")
    print("Missing patient_id:",
          df["patient_id"].isna().sum()
          if "patient_id" in df.columns else "Not applicable")
    print("Top 5 columns with missing values:")
    print(df.isna().sum().sort_values(ascending=False).head(5))


PATIENTS
Rows: 309217
Columns: 6
Exact duplicate rows: 0
Missing booking_id: Not applicable
Missing patient_id: 0
Top 5 columns with missing values:
patient_id      0
patient_name    0
phone           0
dob             0
city            0
dtype: int64

BOOKINGS
Rows: 1252786
Columns: 12
Exact duplicate rows: 20
Missing booking_id: 0
Missing patient_id: 0
Top 5 columns with missing values:
test_code     1290
booking_id       0
created_at       0
slot_at          0
city             0
dtype: int64

COLLECTIONS
Rows: 552492
Columns: 6
Exact duplicate rows: 2894
Missing booking_id: 0
Missing patient_id: Not applicable
Top 5 columns with missing values:
phlebotomist_id    343938
collected_at        79172
booking_id              0
clinic_id               0
channel                 0
dtype: int64

RESULTS
Rows: 1078420
Columns: 5
Exact duplicate rows: 0
Missing booking_id: 0
Missing patient_id: Not applicable
Top 5 columns with missing values:
test_code       3211
booking_id         0
result_t

### 03.3 | Key Uniqueness & Relationships

**Objective:** Verify record identifiers, identify repeated booking IDs, and understand how bookings, collections and results relate.

In [8]:
bookings = datasets["bookings"]
collections = datasets["collections"]
results = datasets["results"]
patients = datasets["patients"]

for name, df, key in [
    ("Patients", patients, "patient_id"),
    ("Bookings", bookings, "booking_id"),
    ("Collections", collections, "booking_id"),
    ("Results", results, "booking_id")
]:
    print(f"\n=== {name.upper()} ===")
    print("Total rows:", len(df))
    print("Unique IDs:", df[key].nunique())
    print("Repeated ID rows:", df[key].duplicated().sum())
    print("Missing IDs:", df[key].isna().sum())

print("\n=== BOOKING-TEST COMBINATIONS ===")
print(
    "Unique booking-test pairs:",
    bookings[["booking_id", "test_code"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Unique result booking-test pairs:",
    results[["booking_id", "test_code"]]
    .drop_duplicates()
    .shape[0]
)

print("\n=== COLLECTION STATUSES ===")
print(collections["status"].value_counts(dropna=False))

print("\n=== RESULT DATE COVERAGE ===")
result_dates = pd.to_datetime(
    results["result_ts"], errors="coerce"
)

print("Earliest:", result_dates.min())
print("Latest:", result_dates.max())
print("Invalid dates:", result_dates.isna().sum())


=== PATIENTS ===
Total rows: 309217
Unique IDs: 309217
Repeated ID rows: 0
Missing IDs: 0

=== BOOKINGS ===
Total rows: 1252786
Unique IDs: 541465
Repeated ID rows: 711321
Missing IDs: 0

=== COLLECTIONS ===
Total rows: 552492
Unique IDs: 549598
Repeated ID rows: 2894
Missing IDs: 0

=== RESULTS ===
Total rows: 1078420
Unique IDs: 473905
Repeated ID rows: 604515
Missing IDs: 0

=== BOOKING-TEST COMBINATIONS ===
Unique booking-test pairs: 1252765
Unique result booking-test pairs: 1078411

=== COLLECTION STATUSES ===
status
collected    473320
no_show       79172
Name: count, dtype: int64

=== RESULT DATE COVERAGE ===
Earliest: 2024-12-31 01:21:00
Latest: 2026-07-02 02:07:00
Invalid dates: 0


## 04 | GROWTH VALIDATION — H1 2025 vs H1 2026

**Objective:** Compare test-result volumes for January–June 2025 and January–June 2026, and evaluate whether the reported 5% growth is supported by the data.

In [9]:
# Diagnostic comparison of result-volume definitions

r = results.copy()
r["result_date"] = pd.to_datetime(r["result_ts"], errors="coerce")

h1_2025 = r[
    (r["result_date"] >= "2025-01-01") &
    (r["result_date"] < "2025-07-01")
]

h1_2026 = r[
    (r["result_date"] >= "2026-01-01") &
    (r["result_date"] < "2026-07-01")
]

for label, data in [
    ("H1 2025", h1_2025),
    ("H1 2026", h1_2026)
]:
    print(f"\n=== {label} ===")
    print("Result rows:", len(data))
    print("Unique bookings:", data["booking_id"].nunique())
    print(
        "Unique booking-test pairs:",
        data[["booking_id", "test_code"]]
        .drop_duplicates()
        .shape[0]
    )

print("\n=== GROWTH COMPARISON ===")

for metric, old, new in [
    ("Result rows", len(h1_2025), len(h1_2026)),
    (
        "Unique booking-test pairs",
        h1_2025[["booking_id", "test_code"]].drop_duplicates().shape[0],
        h1_2026[["booking_id", "test_code"]].drop_duplicates().shape[0]
    )
]:
    growth = ((new - old) / old * 100) if old else float("nan")
    print(f"{metric}: {growth:.2f}%")


=== H1 2025 ===
Result rows: 356442
Unique bookings: 158026
Unique booking-test pairs: 356440

=== H1 2026 ===
Result rows: 378160
Unique bookings: 164333
Unique booking-test pairs: 378156

=== GROWTH COMPARISON ===
Result rows: 6.09%
Unique booking-test pairs: 6.09%
